# Prediction of strokes

While the main objective of this kernel is to produce a predictive model, it will be necessary to perform an exploratory analysis of the data first, in order to properly understand the dataset and select an appropriate algorithm.

I will use the following libraries
* `numpy` for maths
* `pandas` for data manupulation
* `scikit-learn` for machine learning algorithms
* `seaborn` for visualisation
* `random` for sampling
* `scipy` for a distance metric

In [ ]:
%matplotlib inline
import numpy
import numpy.linalg
import pandas 
import sklearn
import sklearn.feature_selection
import sklearn.ensemble
import sklearn.metrics
import random
import seaborn
import scipy.spatial.distance



First import the data

In [ ]:
data = pandas.read_csv('/kaggle/input/healthcare-dataset-stroke-data/train_2v.csv',
                      index_col='id')
data

We need to convert the categorical columns to numerical values

In [ ]:
unique_values = {column:data[column].unique()
                 for column in ('gender','ever_married','work_type','Residence_type','smoking_status')}
unique_values

In [ ]:
for (key,values) in unique_values.items():
    for (i,value) in enumerate(values):
        data.loc[data[key]==value,key] = i
data=data.fillna(0)
data

I will now use Mutual Information to determine which variables are correlated with each other.


In [ ]:
MI=pandas.DataFrame(numpy.zeros((data.shape[1],data.shape[1])),
                    index = data.columns,
                    columns = data.columns)
continuous = ('age','avg_glucose_level','bmi')
for (i,column) in enumerate(data.columns[:-1]):
    later = data.columns[i+1:]
    H = (sklearn.feature_selection.mutual_info_regression 
                                 if column in continuous 
                                 else sklearn.feature_selection.mutual_info_classif)(data[later],data[column])
    MI.loc[column,later] = H
    MI.loc[later,column] = H
seaborn.heatmap(MI)

At first glance, there seem to be few strong preditors of stroke in the dataset. Let's look at the correlations with stroke by themselves.

In [ ]:
MI['stroke'].plot.bar()

The strongest predictor of stroke seems to be age. In fact, looking at the overall Mutual Information chart, age seems to be the only variable that is strongly correlated with anything. Surprisingly, BMI and smoking status do not appear to predict stroke.

I now want to look at how to use the continuous variables. First, let's see how glucose varies with age.

In [ ]:
data.plot.scatter('age','avg_glucose_level')

Very little correlation at all.

Also, let's look at the overall probability of strokes

In [ ]:
data['stroke'].value_counts().plot.bar()

The target variable is highly unbalanced, and only very weakly correlated with any of the other variables. With data like this, ensemble methods are more likely to work than anything else, so I'll try Random Forests. I'll train on a randomly selected 70% of the sample, and test on the remaining 30%.

Because the sample is so unbalanced, it is possible that a random sample may contain too few positive examples to train a meaningful classifier. Therefore, the training sample will be selected as 70% of the negative examples and 70% of the positive examples.

In [ ]:
negative = data[data['stroke']==0].index.values.tolist()
positive = data[data['stroke']==1].index.values.tolist()
training_sample = random.sample(negative,(len(negative)*7)//10)+random.sample(positive,(len(positive)*7)//10)
test_sample = [n for n in data.index.values if n not in training_sample]
cols = [column for column in data.columns if column!='stroke']

model = sklearn.ensemble.RandomForestClassifier(n_estimators=100,
                                                n_jobs=-1,
                                                class_weight='balanced')
model.fit(data.loc[training_sample,cols].values,data.loc[training_sample,'stroke'].values)
predictions = model.predict(data.loc[test_sample,cols].values)
confusion = sklearn.metrics.confusion_matrix(data.loc[test_sample,'stroke'].values,predictions)
seaborn.heatmap(confusion)

Now we will assess the performance of the model

In [ ]:
def precision(cm):
    return cm[1,1]/cm[:,1].sum()

def recall(cm):
    return cm[1,1]/cm[1].sum()

def accuracy(cm):
    return (cm[0,0]+cm[1,1])/cm.sum()

def matthews(cm):
    return (cm[0,0]*cm[1,1]-cm[1,0]*cm[0,1])/numpy.sqrt(cm[0].sum()*cm[1].sum()*cm[:,0].sum()*cm[:,1].sum())

precision(confusion)

16.7% of predicted strokes are correct predictions

In [ ]:
recall(confusion)

 0.04% of actual strokes are correctly predicted

In [ ]:
accuracy(confusion)

Predictions are 98.2% accurate overall

In [ ]:
matthews(confusion)

The models predictions are 2.4% better than guesswork.

Given the lack of strong correlations and the highly unbalanced data, this poor performance is not surprising.

Now let us examine the importance of the features in the fitted models.

In [ ]:
pandas.Series(model.feature_importances_,
             index=cols).plot.bar()

Age is the strongest predictor, followed by average glucose level and BMI. Interestingly, BMI is a stronger predictor than Mutual Information suggested.

Suppose we calculate the mean and covariance of the age, average glucose level and BMI for stroke patients, and then select the non-stroke patients most similar in these metrics to the stroke patients. This will allow us to construct a balanced sample in which the effects of the categorical variables can be assessed.


In [ ]:
stroke_patients = data.loc[data['stroke']==1]
stroke_patients.shape

In [ ]:
continuous_for_stroke = stroke_patients.loc[:,continuous]
mean = continuous_for_stroke.mean(axis=0)
mean.plot.bar()

In [ ]:
covar = continuous_for_stroke.cov()
seaborn.heatmap(covar)

In [ ]:
invcov = numpy.linalg.inv(covar.values)
seaborn.heatmap(invcov)

In [ ]:
non_stroke = data.loc[data['stroke']==0]
distances = pandas.Series(scipy.spatial.distance.cdist(non_stroke.loc[:,continuous].values,
                                                       mean.values.reshape((1,3)),
                                                       'mahalanobis',
                                                        invcov)[:,0],
                          index=non_stroke.index)
closest = distances.nsmallest(783)
closest

In [ ]:
combined = stroke_patients.append(non_stroke.loc[closest.index])
MI=pandas.DataFrame(numpy.zeros((combined.shape[1],combined.shape[1])),
                    index = combined.columns,
                    columns = combined.columns)
continuous = ('age','avg_glucose_level','bmi')
for (i,column) in enumerate(combined.columns[:-1]):
    later = data.columns[i+1:]
    H = (sklearn.feature_selection.mutual_info_regression 
                                 if column in continuous 
                                 else sklearn.feature_selection.mutual_info_classif)(combined[later],combined[column])
    MI.loc[column,later] = H
    MI.loc[later,column] = H
seaborn.heatmap(MI)

Despite the fact that this sample was selected so that age, glucose and BMI would be similar for stroke and non-stroke patients, they still seem to be the strongest predictors of strokes in this sample.

In [ ]:
MI['stroke'].plot.bar()